# From Language Model to Structured Chat

**LLM Agents and Video Analysis · guided lesson**  
**Plan for:** 60–90 minutes  
**Code runtime:** live RCD requests; Clemson network or CUVPN required

## Learning outcomes

- Connect next-token prediction to chat messages and templates.
- Construct role-structured requests and preserve conversation state explicitly.
- Compare local RLS and OpenAI-gateway routing through one SDK.
- Inspect requests without exposing credentials or confusing probability with truth.

| Segment | Suggested minutes |
|---|---:|
| Motivation and mental model | 15–20 |
| Guided implementation | 35–45 |
| Failure analysis and exercise | 15–20 |
| Summary and homework | 5 |
| **Total** | **60–90** |

## Driving question

> **How does an autoregressive continuation model become a multi-turn chat system without gaining hidden memory?**

You need Python and basic API familiarity. The RCD endpoint is the classroom
service, and Clemson network or CUVPN access is required. Each concept below is
followed immediately by the code and evidence used to test it.

## How this lesson builds, step by step

1. **Observe repeated next-token continuations.**
2. **Render ordinary and thinking chat requests.**
3. **Send one text-only request through the RLS OpenAI gateway.**
4. **Store and resend application-owned conversation history.**
5. **Inspect a redacted request and its decoding settings.**

If a result differs from your prediction, stop at that boundary before continuing.

## Work the smallest useful example

Suppose the context contains 120 input tokens, the service permits 256 total tokens, and `max_tokens=80`. The request leaves 56 tokens unused. Appending a 100-token earlier answer raises the prompt to 220, so the same output budget no longer fits. The model did not remember the earlier answer; the application chose whether to resend it. If logits are `[2,1]`, temperature one yields probabilities near `[.73,.27]`; lowering temperature concentrates the same preferences but cannot verify either continuation.


In [ ]:
from pathlib import Path
import os
import sys

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "course_helpers.py").is_file():
        COURSE_ROOT = candidate.resolve()
        break
else:
    raise RuntimeError("Open this notebook from the course directory or its notebooks/ folder.")

os.chdir(COURSE_ROOT)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
print("Course root:", COURSE_ROOT)


## Mental model for this lecture

We will treat the model as a probabilistic function:

`tokens already supplied → scores for possible next tokens`

This deliberately leaves out transformer internals, training, and optimization. It is still powerful enough to explain variation, prompt sensitivity, chat templates, context limits, and why fluent output is not automatically factual.

<details><summary><strong>Reference note: base models and instruction-tuned models</strong></summary>

A base model primarily learns continuation from training text. An instruction-tuned or chat model receives additional training to follow conversational formats and preferences. Both ultimately generate tokens; post-training changes which continuations are likely for structured instructions.

</details>


In [ ]:
import json
from collections.abc import Mapping

def run_completion(prompt, model, client, max_tokens=40, temperature=0.8):
    result = client.completions.create(
        model=model, prompt=prompt,
        max_tokens=max_tokens, temperature=temperature,
    )
    return result.choices[0].text

def run_chat(messages, model, client, max_tokens=500,
             temperature=0.2, enable_thinking=False, tools=None):
    request = dict(model=model, messages=messages,
                   max_tokens=max_tokens, temperature=temperature)
    if tools:
        request["tools"] = tools
    if enable_thinking:
        request["extra_body"] = {
            "chat_template_kwargs": {"enable_thinking": True}
        }
    return client.chat.completions.create(**request).choices[0].message

def message_parts(message):
    row = message if isinstance(message, Mapping) else message.model_dump(exclude_none=True)
    return {
        "reasoning": str(row.get("reasoning_content", row.get("reasoning", "")) or ""),
        "content": str(row.get("content", "") or ""),
    }

def message_content(message):
    return message_parts(message)["content"]

def run_openai_response(prompt, model, client, max_output_tokens=300):
    response = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=max_output_tokens,
    )
    return response.output_text


These functions reveal the entire request contract: input messages, fixed
model ID, output budget, sampling temperature, optional tool schema, and the
extra flag that enables a thinking channel. `message_parts()` deliberately
keeps reasoning text separate from the final answer. A thinking model may
return `reasoning_content`; ordinary models usually do not.

Once these implementations are understood, we use the maintained versions
in `course_helpers.py`. Moving stable code into a module avoids copying and
silently changing it across notebooks.


In [ ]:
from course_helpers import (
    CHAT_MODEL, COMPLETION_MODEL, OPENAI_TEXT_MODEL, THINKING_MODEL,
    create_client, create_openai_gateway_client,
    message_content, message_parts, redact_request, require_models,
    run_chat, run_completion, run_openai_response,
)

client = create_client()
require_models(
    [COMPLETION_MODEL, CHAT_MODEL, THINKING_MODEL],
    client=client,
)
print("Completion model:", COMPLETION_MODEL)
print("Non-thinking chat model:", CHAT_MODEL)
print("Thinking chat model:", THINKING_MODEL)


<details><summary><strong>Code walkthrough: fixing models for a fair comparison</strong></summary>

This lesson does not choose the first catalog entry. It uses the
non-instruction-tuned `gemma-4-31b-non-it` for raw completion,
`qwen3-30b-a3b-instruct-fp8` for ordinary chat, and `qwen3.5-9b` for
thinking chat. The non-IT Gemma model is intentional: a raw completion
prompt does not include the chat template expected by an instruction-tuned
model. `require_models()` checks availability before any teaching request.
`client` is reused so authentication and endpoint configuration remain
identical across calls.

</details>


### Chat still generates one token at a time

The model scores a next token, decoding selects one, and the selected token becomes part of the next input.

**Check your understanding:** Which step is probabilistic, and which state must the application preserve for the next turn?

## 1. Tokens and next-token prediction

Text is first divided into **tokens**—words, word pieces, punctuation, or other learned units. A token is not reliably one word, and token counts differ across languages and content types. The model receives token IDs, not raw human meaning.

A language model assigns scores to possible next tokens. Those scores become a probability distribution; a decoding rule selects one token, appends it, and repeats. Chat does not replace this mechanism—it supplies a learned structure around the token sequence.

<img src="../assets/figures/next-token-loop.svg" alt="Next-token loop from current tokens through model scores and a probability distribution, then back to the input" width="960">

**Important distinctions:**

- **Probability is not confidence in truth.** It measures how plausible a continuation is under the model.
- **Temperature changes sampling sharpness.** It can change variation but does not verify facts.
- **A stop condition ends generation.** This may be a stop token, length limit, or API setting.

**Predict before running:** If we request the same continuation three times, must the outputs be identical? Why?


### Two controls that students should read before every generation call

`temperature` changes how sharply the decoder favors high-scoring tokens.
At a low value such as `0.1`, lower-ranked alternatives are strongly
suppressed; at a higher value such as `0.9`, alternatives have more chance
to be sampled. It changes variation, not factual verification. A high
temperature does not make a model creative in every case, and a low value
does not guarantee identical output unless the backend also provides a
deterministic decoding contract.

`max_tokens` is the maximum number of **new output tokens** requested from
Chat Completions. It is not a word count and it does not enlarge the context
window. Input tokens, generated tokens, and—in a thinking model—reasoning
tokens all consume capacity. If the limit is too small, an answer may stop
midway; if it is unnecessarily large, the request reserves a wider output
budget and may permit a verbose response. We will always name both values
in comparisons so that a model change is not confused with a decoding
change.


## Learn and test: Observe repeated next-token continuations.

Chat APIs accept structured messages, but the model ultimately receives a serialized token sequence. Role markers are part of the learned protocol. A system message can strongly influence behavior, yet it is not an authorization boundary and cannot guarantee compliance. The application owns the message list, decides which history to retain, and chooses how to compact it when the context window becomes crowded.


### A conversation must fit inside one context window

Instructions, history, the current request, and the answer all compete for a bounded token budget.

**Check your understanding:** If the history grows, what can the application summarize or remove without losing the current constraint?

<img src="../assets/figures/context-budget.svg" alt="Course-original figure: Instructions, history, the current request, and the answer all compete for a bounded token budget." width="960">


### Try it: Observe repeated next-token continuations.

**What this cell shows:** Observe repeated next-token continuations.

**What goes in and comes out:** One fixed prompt and decoding settings enter; several continuations are collected.

**Before you run the cell:** Compare the continuations token by token and record whether repeated runs differ under the chosen settings.


In [ ]:
prompt = "The blue color of the daytime sky is caused by "
samples = [
    run_completion(
        prompt, COMPLETION_MODEL, client=client,
        max_tokens=35, temperature=0.8,
    )
    for _ in range(3)
]

for index, sample in enumerate(samples, 1):
    print(f"{index}. {sample.strip()}")

**What you should see:** Variation is possible because each item is a separate live sampled request.

**If your result looks different:** Do not describe one repeated answer as proof of determinism or one variation as a service error.


<details><summary><strong>Code walkthrough: repeated completions</strong></summary>

- The prompt deliberately ends mid-sentence so the completions endpoint continues it.
- The list comprehension sends three independent live requests with the same input.
- `strip()` only cleans surrounding whitespace; it does not normalize or judge the answer.

The loop prints multiple samples because comparing a distribution of behavior is more informative than treating one response as canonical.

</details>


### Exercise — interpret variation

Identify what remains semantically stable across the samples and what changes. Nondeterminism means we should assess behavior and evidence rather than exact wording.


## 2. From a prompt to structured chat

Chat APIs serialize messages with roles. The system message establishes instructions; user and assistant messages form the conversation. Instructions influence behavior but are not a security boundary.

Conceptually, the API applies the model's **chat template** to serialize roles and content into one token sequence. Different model families may use different control tokens, which is why applications should use the supported chat endpoint/template instead of inventing a raw format.

Role semantics:

- `system`: application-level instructions and constraints;
- `user`: the current request or supplied content;
- `assistant`: prior model responses and tool-call proposals;
- `tool`: results returned by application code after a tool call.

The role label establishes structure, not absolute priority enforcement. Untrusted text can still influence model behavior.

**Predict before running:** Which message would you change to request an explanation for a middle-school audience?


## Learn and test: Render ordinary and thinking chat requests.

Base models, completion endpoints, and instruction-tuned chat models expose related but different contracts. A continuation endpoint reveals next-token behavior directly. A chat endpoint usually applies a template matching post-training. Sending a raw transcript with invented role labels can produce different token IDs from the official template. Reproducible comparisons freeze the model ID, message structure, temperature, token budget, stopping rules, and seed when supported.


### Follow data across the application boundary

The diagram separates local notebook work, the request sent to the service, and application-owned validation.

**Check your understanding:** Which information crosses the network, and which files remain local unless the code explicitly sends them?

<img src="../assets/figures/course-data-flow.svg" alt="Course-original figure: The diagram separates local notebook work, the request sent to the service, and application-owned validation." width="960">


### Try it: Render ordinary and thinking chat requests.

**What this cell shows:** Render ordinary and thinking chat requests.

**What goes in and comes out:** The same messages enter two fixed models; final content plus the presence and length of a separate reasoning field come out.

**Before you run the cell:** Confirm that prompt and temperature are fixed and that the thinking model has a sufficient output budget.


In [ ]:
# Prepare the inputs and settings used below.
messages = [
    {"role": "system", "content": "You are a concise science tutor. Separate observation from inference."},
    {"role": "user", "content": "Why does the daytime sky look blue?"},
]
response = run_chat(
    messages, CHAT_MODEL, client=client,
    max_tokens=180, temperature=0.2,
)
print("Non-thinking answer:\n", message_content(response))

thinking_response = run_chat(
    messages, THINKING_MODEL, client=client,
    max_tokens=600, temperature=0.2,
    enable_thinking=True,
)
thinking = message_parts(thinking_response)
print("\nThinking field returned:", bool(thinking["reasoning"]))
print("Thinking characters (not displayed):", len(thinking["reasoning"]))
print("Thinking-model final answer:\n", thinking["content"])

**What you should see:** Both final answers are visible while reasoning content remains separate and undisplayed.

**If your result looks different:** If final content is empty, inspect `reasoning_content`, `content`, and the output-token budget separately.


<details><summary><strong>Code walkthrough: building a chat request</strong></summary>

`messages` is ordered: the system instruction comes first and the user's question follows. Both requests keep the prompt and temperature fixed. The thinking request receives a larger `max_tokens` budget because reasoning tokens can use part of that budget.

`message_content()` returns only the final `content`. For the thinking model,
`message_parts()` also checks `reasoning_content` (and the older `reasoning`
spelling) without mixing it into the answer. The notebook reports whether
reasoning was returned and its length, but does not print or grade private
reasoning text. Applications should evaluate the final answer and external
evidence, not treat hidden reasoning as a correctness certificate.

Notice that the API key is not inside `messages`. Authentication belongs to the HTTP client; instructions and user data belong to the message payload.

</details>


## 3. Use the same SDK with the RLS OpenAI gateway

The `openai` Python package is a client library, not a guarantee that every
request goes directly to `api.openai.com`. The client's `base_url` selects
the service:

- `https://llm.rcd.clemson.edu/v1` routes requests to locally hosted RLS models;
- `https://llm.rcd.clemson.edu/openai/v1` routes eligible requests through
  RLS to OpenAI-hosted models.

The second route has separate model availability, credit accounting, and
policy checks. RLS currently blocks image/file inputs through this gateway,
so this lesson uses a small **text-only** Responses API request. Video stays
in Lecture 3 with the locally hosted Qwen3-Omni model.

Request content sent through the OpenAI gateway is forwarded to OpenAI.
Use only data permitted by Clemson, sponsor, IRB, and data-use requirements.

**Before running:** predict which client setting changes the destination.
The prompt and Python package are not what select the endpoint.


## Learn and test: Send one text-only request through the RLS OpenAI gateway.

Identical outputs do not prove determinism because a dominant distribution can repeatedly select the same text. Evaluate task behavior across fixed prompts and several samples. Treat fluent statements as model output requiring verification rather than confidence estimates.


### Structured roles become one serialized model input

The application keeps messages as role/content records, then the tokenizer's chat template turns them into one token sequence.

**Check your understanding:** Which boundary preserves the message roles, and where would you inspect the exact tokens sent to the model?

<img src="../assets/figures/chat-roles.svg" alt="Course-original figure: The application keeps messages as role/content records, then the tokenizer's chat template turns them into one token sequence." width="960">


### Try it: Send one text-only request through the RLS OpenAI gateway.

**What this cell shows:** Send one text-only request through the RLS OpenAI gateway.

**What goes in and comes out:** A plain-text prompt and the OpenAI gateway base URL enter; a text response from an OpenAI-hosted model comes out.

**Before you run the cell:** Confirm the gateway URL, exact model ID, text-only input, and output budget.


In [ ]:
openai_client = create_openai_gateway_client()
require_models([OPENAI_TEXT_MODEL], client=openai_client)

openai_answer = run_openai_response(
    "Explain in two sentences why an API base URL matters.",
    model=OPENAI_TEXT_MODEL,
    client=openai_client,
    max_output_tokens=180,
)
print("Gateway URL:", openai_client.base_url)
print("OpenAI-hosted model:", OPENAI_TEXT_MODEL)
print(openai_answer)

**What you should see:** The response demonstrates that one SDK can target separately governed endpoints.

**If your result looks different:** A successful local-model request does not prove that gateway credits or policy permit the OpenAI request.


<details><summary><strong>Code walkthrough: one SDK, two endpoints</strong></summary>

`create_openai_gateway_client()` uses the same hidden RLS API key but changes
`base_url` to the OpenAI gateway. `require_models()` checks the gateway's own
catalog, not the local-model catalog. `run_openai_response()` then calls the
Responses API with plain text and reads `response.output_text`.

A `PermissionDeniedError` here means RLS rejected the request before or while
forwarding it. Check the OpenAI Credits page for the recorded denial reason:
common causes include missing allocation, exhausted credits, an unavailable
model, or a request type blocked by gateway policy.

</details>


### Exercise — change the audience

Edit only the system message below, then compare tone, vocabulary, and length. Do not expect an exact phrase.


## Learn and test: Store and resend application-owned conversation history.

Request inspection should expose roles, lengths, model ID, and decoding configuration while redacting secrets and sensitive content. Never distribute executed notebooks containing API keys, private prompts, or reasoning text.


### Try it: Store and resend application-owned conversation history.

**What this cell shows:** Store and resend application-owned conversation history.

**What goes in and comes out:** A message list and one new user turn enter; the explicitly extended history is sent again.

**Before you run the cell:** Print the role sequence and confirm that the earlier constraint is present in the new request.


In [ ]:
audience_messages = [
    {"role": "system", "content": "Explain science to a middle-school audience in two sentences."},
    {"role": "user", "content": "Why does the daytime sky look blue?"},
]
audience_response = run_chat(
    audience_messages, CHAT_MODEL, client=client,
    max_tokens=120, temperature=0.2,
)
print(message_content(audience_response))

**What you should see:** The model uses earlier content only because the application includes it again.

**If your result looks different:** If a prior fact is absent from the request, do not treat its omission from the answer as hidden-memory failure.


## 4. Conversation state belongs to the application

The model does not remember this notebook automatically. Each request is evaluated from the content sent with that request. An application creates apparent memory by resending selected messages, retrieved records, or a compacted summary.

This creates three design responsibilities:

1. **Selection:** include what is relevant to the current task.
2. **Privacy:** avoid adding secrets or unrelated personal context.
3. **Integrity:** do not let untrusted content silently rewrite goals or permissions.


## Learn and test: Inspect a redacted request and its decoding settings.




### Try it: Inspect a redacted request and its decoding settings.

**What this cell shows:** Inspect a redacted request and its decoding settings.

**What goes in and comes out:** One request is copied for display after secret fields are replaced.

**Before you run the cell:** Confirm that model ID, roles, temperature, and max_tokens remain visible while authorization is redacted.


In [ ]:
history = [
    {"role": "system", "content": "You are a concise science tutor."},
    {"role": "user", "content": "Why does the daytime sky look blue?"},
    {"role": "assistant", "content": message_content(response)},
    {"role": "user", "content": "State the key mechanism in five words or fewer."},
]
follow_up = run_chat(
    history, CHAT_MODEL, client=client,
    max_tokens=40, temperature=0.2,
)
print(message_content(follow_up))

**What you should see:** The preview supports debugging without exposing the API key.

**If your result looks different:** If the bearer value is visible, stop and fix redaction before saving or sharing the notebook.


## Inspect safely

Request inspection is useful for debugging, but secrets must be redacted. The helper recursively masks common secret fields.

API metadata such as model name, token limits, and message roles is normally safe to inspect. Authorization headers, API keys, passwords, and private video contents are not debugging decorations; minimize and redact them.


In [ ]:
request_preview = {
    "base_url": "https://llm.rcd.clemson.edu/v1",
    "Authorization": "Bearer example",
    "model": CHAT_MODEL,
    "messages": messages,
    "temperature": 0.2,
    "max_tokens": 180,
}
print(json.dumps(redact_request(request_preview), indent=2))

## Checkpoint

1. Chat is still next-token prediction over a specially formatted input.
2. Roles help organize instructions and history; they do not guarantee obedience or truth.
3. Your application chooses what history to resend.
4. Evaluate concepts and evidence, not exact output strings.


## Optional extension — temperature

Repeat the same benign prompt at `temperature=0.1` and `temperature=0.9`
while holding the model, messages, and `max_tokens` fixed. Collect several
samples at each setting; one sample is not enough to characterize variation.


## A realistic failure to investigate

A common failure is believing conversation state lives inside the endpoint. Omitting an earlier constraint from the next request causes the model to lose it, while sending the entire transcript can exceed context or preserve superseded instructions. Another failure is treating a system prompt as security enforcement. Keep authorization in application code and record exactly which messages were sent.


## Guided exercise

Modify one supplied fixture to trigger a realistic failure. Predict which validator or policy layer should catch it, run the reduced path, and report the observed category plus one remaining risk.

## Check your work

Try the exercise before expanding the reference solution.

In [ ]:
print('Expected method: change one fixture, preserve mode and policy, and identify the earliest rejecting boundary.')

## Final concept map

User goal → structured roles → chat template/token sequence → model logits → decoding → returned message → application appends selected history. Context and state belong to the application; probability and truth remain different questions.


## Homework

Run one controlled live comparison or agent/video failure matrix. Submit configuration without secrets, structured outcomes, one success, one failure, and a limitation. The full rubric is in `homework/`.

## Glossary

| Term | Meaning in this lesson |
|---|---|
| **token** | A vocabulary unit represented by an integer ID. |
| **logit** | An unrestricted score for a candidate token. |
| **temperature** | A decoding control that rescales logits. |
| **message** | A role/content record retained by the application. |
| **chat template** | The tokenizer-specific serialization of structured messages. |
| **context window** | The bounded token capacity available to one request. |
| **conversation state** | History or summary explicitly stored and resent by the application. |
| **redaction** | Removing secrets or sensitive fields before display or logging. |


## Sources and further study

- [Clemson RCD LLM Service](https://docs.rcd.clemson.edu/llm/)
- [OpenAI-compatible local model API](https://docs.rcd.clemson.edu/llm/usage/api/)
- [Clemson acceptable-use guidance](https://docs.rcd.clemson.edu/llm/acceptable_use/)
- [OpenAI API through the RLS gateway](https://docs.rcd.clemson.edu/llm/usage/openai_api/)
- [OpenAI GPT-5.6 model documentation](https://developers.openai.com/api/docs/models)

Course prose, code, and SVG diagrams are original. The video is NASA SVS item 30628 and is credited in the asset manifest. Sources verify service contracts and responsible-use terminology.
